# 🛡️ Academic Benchmark: Multimodal Threat Stacking Ensemble
### BTech AIML Capstone Project: Level 1 Meta-Learner Stacking vs. Weighted Late Fusion

This notebook demonstrates the complete, scientifically rigorous implementation of the **Stacking Ensemble Fusion** architecture for the Multimodal Threat Prediction Engine.

**Architecture Hierarchy:**
- **Level 0 (Base Models):**
  1. Physiological Biometrics Model ($P_{\text{physio}} = P(\text{Stress} \mid \text{WESAD Biometrics})$)
  2. Geospatial Campus Vulnerability Engine ($P_{\text{spatial}} \in [0, 1]$ via BIT Mesra Geofence & Haversine Distance)
  3. Environmental Context Evaluator ($P_{\text{env}} \in [0, 1]$ via Campus Local Time, Illumination & Crowd Density)
- **Level 1 (Meta-Learner):**
  - Logistic Regression Meta-Learner: $\text{logit}(P_{\text{final}}) = \beta_0 + \beta_1 P_{\text{physio}} + \beta_2 P_{\text{spatial}} + \beta_3 P_{\text{env}}$
  - Benchmarked against Random Forest Stacker, XGBoost Stacker, and the Weighted Late Fusion Baseline.

In [1]:
import sys
from pathlib import Path

# Ensure project root is prioritized
project_root = Path("..").resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import config
from src.fusion.train import run_academic_benchmark_experiment
from src.fusion.stacking_pipeline import StackingEnsemblePipeline
from src.fusion.predict import MultimodalThreatPredictor

## 1. Run Academic Stacking Benchmark Experiment
Executes leakage-free Out-of-Fold (OOF) cross-validation and evaluates on an untouched, subject-grouped test set.

In [2]:
results = run_academic_benchmark_experiment(n_samples=4000, test_ratio=0.20, random_state=42)

## 2. Benchmark Comparison Results Table
Displays comparative evaluation metrics across all 7 candidates on the untouched test partition.

In [3]:
import pandas as pd
from src.fusion.evaluation import format_comparison_table

df_cmp = format_comparison_table(results)
df_cmp

## 3. Visualizations: Performance & Calibration Reliability Curves

In [4]:
from IPython.display import Image, display

print("Model Performance Comparison (ROC-AUC, F1, Brier Score, ECE):")
display(Image(filename="plots/stacking_comparison_bar.png"))

print("Probability Calibration Reliability Curves (Untouched Test Partition):")
display(Image(filename="plots/stacking_calibration_curves.png"))

## 4. Model Explainability & Learned Meta-Learner Parameters

In [5]:
pipeline = StackingEnsemblePipeline.load_artifact()
if pipeline and pipeline.is_trained:
    explain_dict = pipeline.meta_learner.explain()
    print("Learned Meta-Learner Parameters:")
    print(f"  Intercept (beta_0): {explain_dict['intercept_beta0']:+.4f}")
    print("  Coefficients:")
    for k, v in explain_dict['coefficients'].items():
        or_val = explain_dict['odds_ratios'][k]
        print(f"    - {k}: beta={v:+.4f} (Odds Ratio={or_val:.2f})")
    print(f"\nInterpretation:\n  {explain_dict['interpretation']}")
    print(f"\nScientific Note:\n  {explain_dict['scientific_note']}")